<!-- nav -->
[← 8 · C Types](08_07_C_Types.ipynb) · [Home](../../README.md) · [8.09 · Cpp Types →](08_09_Cpp_Types.ipynb)

# 8.08 · C Preprocessor

Before C or C++ reads a line of code, its preprocessor reads the directives among them: `#include <stdio.h>`,
`#define WIDTH 8`, `#if defined(X) && X > 1` ... `#endif` (ISO/IEC 9899, 6.10; ISO/IEC 14882, [cpp]). The two standards
define the same directives, and one dialect models them for both, `C Preprocessor`: its directives, `defined(X)`
within their conditions, and the comments kept among them, as mbse-programs has them. A program's directives are
mbse-programs' to hold, among its declarations; here a directive is held by its own holder, `defined(X)` by C's and
C++'s expressions, and a directive or a comment by an enumeration's body, among its enumerators
([8.07 · C Types](08_07_C_Types.ipynb)). This page builds a directive, then each kind of directive, then a conditional
one with its branches; then what a condition binds, what validation says, where C and C++ read its terms, and the
editions it covers.

Like the rest of Part I, it starts from `dialects()`. `S` is its holder namespace, `amd64` the ABI a C family's
holder names (tutorial 2), `variable` writes a name in C's terms (tutorial 4), and `name` an identifier, the name a
directive defines or tests, each a spec the builders call:

In [1]:
from mbse.Expressions.C import Expressions as C, Preprocessor as CPreprocessor, Types as CTypes
from mbse.Expressions.Cpp import Expressions as Cpp
from mbse.Expressions.Framework import Standards, Symbolics, Terms
from toolkit import dialects

store = dialects()
S = store.Expressions
amd64 = store.Instances.Expressions.SystemV.AMD64
variable = lambda word: lambda d: d.variable(lambda v: v.name(word))
name = lambda word: lambda n: n.spelling(word)
DIALECT = CPreprocessor.Expressions.DIALECT


def refused(evaluate, expression):
    """What `evaluate` says of `expression`: its value, or the reason it refuses it."""
    try:
        return evaluate(expression)
    except (NotImplementedError, ValueError) as error:
        return str(error)

<a id="T08.08.1"></a>
## 08.08.1 · A directive, `#define WIDTH 8`

```text
#define WIDTH 8
```

`#define WIDTH 8` defines an object-like macro: a `definedirective`, whose `name` is the identifier it defines, C's
`Identifier` helper, and whose `replacement` is the rest of its line, kept as written. A replacement is a sequence of
tokens, not an expression: the preprocessor substitutes it wherever `WIDTH` is written, before C or C++ parses
anything, so the dialect keeps it as text. Its holder, `S.C.Preprocessor.Expression`, names an ABI as every C family
holder does:

In [2]:
with store.transaction():
    width = S.C.Preprocessor.Expression(lambda e: e.root(
        lambda u: u.definedirective(lambda d: d.name(name("WIDTH")).replacement("8"))).abi([amd64]))
term = Terms.term(width)
form = Terms.form(term)
print(CPreprocessor.Text.ToText(width), "|", form.kind, "of", Terms.dialect_of(term))
print("its name:", form.attributes["name"], "| its replacement:", form.attributes["replacement"])
print(DIALECT.name(), DIALECT.tag, "| its kinds:", ", ".join(DIALECT.kinds()))
assert CPreprocessor.Text.ToText(width) == "#define WIDTH 8"

#define WIDTH 8 | definedirective of Expressions.C.Preprocessor.Term
its name: Part(kind='identifier', attributes={'spelling': 'WIDTH'}, size=0) | its replacement: 8
C Preprocessor c_preprocessor | its kinds: includedirective, definedirective, ifdirective, ifdefdirective, elifdirective, elifdefdirective, elsedirective, otherdirective, definedexpression, comment, error, missing


<a id="T08.08.2"></a>
## 08.08.2 · Each directive

```text
#include <stdio.h>        #include "a.h"        #define MAX(a, b) ((a) > (b) ? (a) : (b))
#define LOG(...) f(__VA_ARGS__)        #undef N        #pragma once        // note
```

Each directive is a kind of its own, with what its grammar writes as properties: an `includedirective`'s `path`, and
whether it is a `system` header, between angle brackets; a function-like `definedirective`'s `parameters`, identifiers,
and whether it is `variadic`; and an `otherdirective` for the directives that hold only their text, `#undef`,
`#pragma`, `#line`, `#error`, named by its `directive`. A comment is a kind too, `comment`, a line comment or a block
one, kept where it was written:

In [3]:
with store.transaction():
    directives = [S.C.Preprocessor.Expression(lambda e, spec=spec: e.root(spec).abi([amd64])) for spec in [
        lambda u: u.includedirective(lambda d: d.path("stdio.h").system(True)),
        lambda u: u.includedirective(lambda d: d.path("a.h")),
        lambda u: u.definedirective(lambda d: d.name(name("MAX")).function_like(True)
            .parameters([name("a"), name("b")]).replacement("((a) > (b) ? (a) : (b))")),
        lambda u: u.definedirective(lambda d: d.name(name("LOG")).function_like(True).variadic(True)
            .replacement("f(__VA_ARGS__)")),
        lambda u: u.otherdirective(lambda d: d.directive("undef").text("N")),
        lambda u: u.otherdirective(lambda d: d.directive("pragma").text("once")),
        lambda u: u.comment(lambda c: c.text(" note")),
        lambda u: u.comment(lambda c: c.text(" a block ").block(True))]]
for directive in directives:
    form = Terms.form(Terms.term(directive))
    print(f"{CPreprocessor.Text.ToText(directive):42}", f"{form.kind:16}", ", ".join(form.attributes))
assert [CPreprocessor.Text.ToText(d) for d in directives] == [
    "#include <stdio.h>", '#include "a.h"', "#define MAX(a, b) ((a) > (b) ? (a) : (b))", "#define LOG(...) f(__VA_ARGS__)",
    "#undef N", "#pragma once", "// note", "/* a block */"]

#include <stdio.h>                         includedirective path, system
#include "a.h"                             includedirective path
#define MAX(a, b) ((a) > (b) ? (a) : (b))  definedirective  name, function_like, parameters, replacement
#define LOG(...) f(__VA_ARGS__)            definedirective  name, function_like, variadic, replacement
#undef N                                   otherdirective   directive, text
#pragma once                               otherdirective   directive, text
// note                                    comment          text
/* a block */                              comment          block, text


<a id="T08.08.3"></a>
## 08.08.3 · A conditional directive and its branches

```text
#if defined(X) && X > 1
A
#elifndef Z
B
#else
// none
#endif
```

A conditional directive includes the lines it holds when its condition holds. `#if`'s `condition` is a term of C's or
C++'s, here C's `&&` of `defined(X)`, the preprocessor's `definedexpression`, which C's `Term` lists among its own kinds,
and `X > 1`. Its `items` are what it includes: enumerators, directives and comments, terms of whichever dialect holds
them, `C Types`' `enumerator` here. Its `alternative` is the branch after it, `#elif`, `#elifdef`, `#elifndef` (an
`elifdefdirective` whose `negated` is set) or `#else`, each holding the next, so the chain is one tree, which the
printer closes with one `#endif`. A form gives the condition and the next branch as its subterms, then the items:

In [4]:
enumerator = lambda word: lambda t: t.c_types(lambda u: u.enumerator(lambda r: r.name(name(word))))
one = lambda t: t.literal(lambda r: r.value(lambda v: v.as_scalar(1)))
condition = lambda t: t.c(lambda d: d.binaryexpression(lambda b: b
    .left(lambda l: l.definedexpression(lambda x: x.name(name("X")))).operator("&&")
    .right(lambda r: r.binaryexpression(lambda g: g.left(variable("X")).operator(">").right(one)))))
with store.transaction():
    conditional = S.C.Preprocessor.Expression(lambda e: e.root(lambda u: u.ifdirective(lambda d: d
        .condition(condition).items([enumerator("A")])
        .alternative(lambda a: a.elifdefdirective(lambda f: f.negated(True).name(name("Z")).items([enumerator("B")])
            .alternative(lambda a: a.elsedirective(lambda s: s.items([
                lambda t: t.c_preprocessor(lambda u: u.comment(lambda c: c.text(" none")))]))))))).abi([amd64]))
print(CPreprocessor.Text.ToText(conditional))
for t in Terms.form(Terms.term(conditional)).arguments:
    print(" ", f"{Terms.kind_of(t)[0]:16}", f"of {DIALECT.owner(t).name():15}", CPreprocessor.Text.ToText(t).replace("\n", " | "))
assert CPreprocessor.Text.ToText(conditional) == "#if defined(X) && X > 1\nA\n#elifndef Z\nB\n#else\n// none\n#endif"

#if defined(X) && X > 1
A
#elifndef Z
B
#else
// none
#endif
  binaryexpression of C               defined(X) && X > 1
  elifdefdirective of C Preprocessor  #elifndef Z | B | #else | // none | #endif
  enumerator       of C Types         A


<a id="T08.08.4"></a>
## 08.08.4 · What a condition binds, and what is checked

A condition's names are macros' names, which the preprocessor replaces before it evaluates the condition: they are
not a term's references, so a conditional directive binds them within its condition, and `X` needs nothing from its
scope (tutorial 7). It binds them nowhere else: an enumerator its branch holds, whose value reads `X`, reads a
variable, which its scope must give. `macros` gives the names a condition reads as macros, `defined(X)`'s among them,
whose name is a macro's, never a reference:

In [5]:
with store.transaction():
    valued = S.C.Preprocessor.Expression(lambda e: e.root(lambda u: u.ifdirective(lambda d: d.condition(condition)
        .items([lambda t: t.c_types(lambda u: u.enumerator(lambda r: r.name(name("A")).value(variable("X"))))]))).abi([amd64]))
print("macros:", ", ".join(sorted(CPreprocessor.Expressions.macros(Terms.form(Terms.term(conditional)).arguments[0]))))
for directive in (conditional, valued):
    print(CPreprocessor.Text.ToText(directive).replace("\n", " | "), "|", "; ".join(DIALECT.validate(directive)) or "valid",
          "| needs:", ", ".join(sorted(Symbolics.free(directive, DIALECT))) or "nothing")
assert DIALECT.validate(conditional) == [] and DIALECT.validate(valued) == ["argument 0: value: variable 'X' is not bound"]

macros: X
#if defined(X) && X > 1 | A | #elifndef Z | B | #else | // none | #endif | valid | needs: nothing
#if defined(X) && X > 1 | A = X | #endif | argument 0: value: variable 'X' is not bound | needs: X


Validation checks each directive as its grammar has it, in mbse-programs' words: an `#include` names a path or a macro,
a directive's names are identifiers, a branch after another is an `#elif`, an `#elifdef` or an `#else`. Text that does
not parse is kept as written, as `error` ([PRI-002: preserve, don't lose](https://github.com/pitaman71/mbse-basis/blob/main/docs/PRINCIPLES.md#PRI-002)),
printed back and reported:

In [6]:
with store.transaction():
    wrong = {source: S.C.Preprocessor.Expression(lambda e, spec=spec: e.root(spec).abi([amd64])) for source, spec in [
        ("#include", lambda u: u.includedirective(lambda d: d)),
        ("#define 1A", lambda u: u.definedirective(lambda d: d.name(name("1A")))),
        ("#ifdef A ... #ifdef B", lambda u: u.ifdefdirective(lambda d: d.name(name("A"))
            .alternative(lambda a: a.ifdefdirective(lambda f: f.name(name("B")))))),
        ("#if (", lambda u: u.error(lambda r: r.text("#if ("))),
    ]}
for source, directive in wrong.items():
    print(f"{source:22}", "|", "; ".join(DIALECT.validate(directive)))
assert DIALECT.validate(wrong["#include"]) == ["an IncludeDirective has a path or a macro"]
assert CPreprocessor.Text.ToText(wrong["#if ("]) == "#if ("

#include               | an IncludeDirective has a path or a macro
#define 1A             | '1A' is not an identifier
#ifdef A ... #ifdef B  | an IfdefDirective's alternative is an elifdirective, an elifdefdirective or an elsedirective
#if (                  | invalid syntax '#if ('


<a id="T08.08.5"></a>
## 08.08.5 · Read by C and C++, and by their types

```text
defined(A) || defined(B)        enum E {
                                    A,
                                #ifdef WIDE
                                    B,
                                #endif
                                    C
                                }
```

C and C++ read `C Preprocessor`'s terms where their grammars put them. `definedexpression` stands among C's and C++'s
own kinds, so `defined(A) || defined(B)` is a C expression, or a C++ one, whose operands are the preprocessor's terms;
and its directives and comment stand among `C Types`' and `Cpp Types`' kinds, so an enumeration's body holds them
between its enumerators, printed at the start of their lines by the family's one printer (tutorial 8, 08.5). Neither
evaluates them: a directive is the preprocessor's to carry out, before evaluation, and each family's evaluator refuses
one by its dialect's name, wherever it stands:

In [7]:
defined = lambda word: lambda d: d.definedexpression(lambda x: x.name(name(word)))
with store.transaction():
    either = {
        "C": (C, S.C.Expressions.Expression(lambda e: e.root(lambda u: u.binaryexpression(lambda b: b
            .left(defined("A")).operator("||").right(defined("B")))).abi([amd64]))),
        "Cpp": (Cpp, S.Cpp.Expressions.Expression(lambda e: e.root(lambda u: u.binaryexpression(lambda b: b
            .left(defined("A")).operator("||").right(defined("B")))).abi([amd64]))),
    }
    body = S.C.Types.Expression(lambda e: e.root(lambda u: u.typeid(lambda i: i.specifiers([
        lambda s: s.enumspecifier(lambda n: n.key("enum").name(lambda m: m.identifier(name("E"))).body(True).enumerators([
            lambda t: t.enumerator(lambda r: r.name(name("A"))),
            lambda t: t.ifdefdirective(lambda d: d.name(name("WIDE")).items([enumerator("B")])),
            lambda t: t.enumerator(lambda r: r.name(name("C")))]))]))).abi([amd64]))
for language, (family, expression) in either.items():
    operands = Terms.form(Terms.term(expression)).arguments
    print(f"{language:4}", family.Text.ToText(expression), "|", ", ".join(
        f"{Terms.kind_of(a)[0]} of {family.Expressions.DIALECT.owner(a).name()}" for a in operands),
        "|", refused(family.Evaluators.OfAny, expression))
print(CTypes.Text.ToText(body))
directive = Terms.form(Terms.term(body)).arguments[2]
print(Terms.kind_of(directive)[0], "of", CTypes.Expressions.DIALECT.owner(directive).name(), "|",
      refused(C.Evaluators.OfAny, conditional))
assert C.Text.ToText(either["C"][1]) == Cpp.Text.ToText(either["Cpp"][1]) == "defined(A) || defined(B)"
assert CTypes.Text.ToText(body) == "enum E {\n    A,\n#ifdef WIDE\n    B,\n#endif\n    C\n}"

C    defined(A) || defined(B) | definedexpression of C Preprocessor, definedexpression of C Preprocessor | C Preprocessor definedexpression is not evaluated
Cpp  defined(A) || defined(B) | definedexpression of C Preprocessor, definedexpression of C Preprocessor | C Preprocessor definedexpression is not evaluated
enum E {
    A,
#ifdef WIDE
    B,
#endif
    C
}
ifdefdirective of C Preprocessor | C Preprocessor ifdirective is not evaluated


<a id="T08.08.6"></a>
## 08.08.6 · Its editions

One dialect for two languages covers both languages' editions, C's three and C++'s three, each by its designation and
its common name ([CAP-268: standards as tagged data](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-268.md)).
Every kind is in all six but one: `#elifdef` and `#elifndef` came with C23 and C++23, so `elifdefdirective` is recorded in
those two alone, and `Standards.of` reads, for a kind, the editions that have it:

In [8]:
pre = store.Instances.Expressions.C.Preprocessor  # the dialect, by its global name
print(", ".join(f"{designation} ({Standards.tags(edition)[1]})" for designation, edition in Standards.standards(pre).items()))
kinds = {b.name: b.schema for b in store.registered(DIALECT.union).branches}
for kind in ("ifdirective", "elifdefdirective"):
    print(f"{kind:16}", ", ".join(Standards.tags(edition)[1] for edition in Standards.of(pre, kinds[kind]).values()))
assert [Standards.tags(e)[1] for e in Standards.of(pre, kinds["elifdefdirective"]).values()] == ["C23", "C++23"]

ISO/IEC 9899:2011 (C11), ISO/IEC 9899:2018 (C17), ISO/IEC 9899:2024 (C23), ISO/IEC 14882:2017 (C++17), ISO/IEC 14882:2020 (C++20), ISO/IEC 14882:2024 (C++23)
ifdirective      C11, C17, C23, C++17, C++20, C++23
elifdefdirective C23, C++23


## What you learned

- **The preprocessor is one dialect for C and C++**, `C Preprocessor`: its directives, `defined(X)` and comments, each
  a kind with what its grammar writes, a macro's replacement kept as written.
- **A conditional directive is one tree**: its condition a C or C++ term, its items any dialect's terms, each branch
  after it its `alternative`.
- **A condition binds its macros' names**, within itself alone.
- **C and C++ read its terms where their grammars put them**: `defined(X)` among their expressions, directives and
  comments among their enumerators; none is evaluated.
- **Its editions are both languages'**, `#elifdef` C23's and C++23's.

Next, C++'s own type-ids, [8.09 · Cpp Types](08_09_Cpp_Types.ipynb).

<!-- nav -->
[← 8 · C Types](08_07_C_Types.ipynb) · [Home](../../README.md) · [8.09 · Cpp Types →](08_09_Cpp_Types.ipynb)